# 02 — End-to-end smoke test + throughput gate (Day 2)

Mandatory milestone (§10.1): a 2-epoch C1 run through the WHOLE
pipeline — staging → dataloader → training → checkpoint on Drive →
resume → metrics CSV — plus the **measured** numbers the budget
depends on: s/step (CE, batch 256) and staging time.

**Pre-committed go/no-go rules (§10.1 — decided before measuring):**
- projected C1 (40 ep × 400 steps) **> 4 h** → escalation §5.2 before any run
- projected phase A (60 ep × 400 steps, ~2× per-step cost for 512 views) **> 8 h** → escalation §5.2
- staging **> ~15 min** → repack fp16 (§8.5)

Escalation order (§5.2): (a) 400→300 steps/epoch; (b) `escalation_b=True`
(stride (2,2) at layer3, ~0.63× FLOPs); (c) fewer epochs with the
pre-committed checkpoint-grid rule. Any escalation goes in the split
file changelog and recalibrates §8.4.

Thin runner: all logic lives in `sharp_har/`. The full
checkpoint→CSV harness (with test-invocation logging) lands on day 3;
here the CSV evidence is `history.csv` written by `train_run`.

## 1. Environment setup

In [1]:
import copy
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/FilippoIsoni/sharp-har.git"
REPO_DIR = Path("/content/sharp-har")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)

# After the clone, so the file actually exists on a fresh runtime (the
# old order made this a silent no-op and the run used the stock env).
!pip install -q -r {REPO_DIR}/requirements.txt

import torch

sys.path.insert(0, str(REPO_DIR))

from sharp_har.utils import set_seed, get_git_hash, read_yaml, write_json

set_seed(42)
assert torch.cuda.is_available(), "the throughput gate is meaningless without the GPU — Runtime > Change runtime type > T4"
print("git hash:", get_git_hash(REPO_DIR))
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

git hash: 1e65c1252740601dee5f06190567fe3fc047ec3d
GPU: Tesla T4 | torch 2.11.0+cu128


## 2. Mount Drive + staging (timed — a gate measurement)

Skipped if the data is already staged in this session (e.g. after
running notebook 01). The staging time below then comes from a
fresh-session run — record THAT number, not a skip.

In [2]:
import time
import zipfile
from google.colab import drive

drive.mount("/content/drive")

paths_cfg = read_yaml(REPO_DIR / "configs" / "paths.yaml")
drive_root = Path(paths_cfg["drive_root"])
stage_dir = Path(paths_cfg["stage_dir"])

already_staged = stage_dir.exists() and any(stage_dir.rglob("*.txt"))
if already_staged:
    staging_seconds = None
    print("data already staged — staging time NOT measured in this session")
else:
    stage_dir.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    for zip_name in paths_cfg["zips"]:
        src = drive_root / zip_name
        dst = Path("/content") / zip_name
        print(f"copying {src} -> {dst}")
        subprocess.run(["cp", str(src), str(dst)], check=True)
        with zipfile.ZipFile(dst) as zf:
            zf.extractall(stage_dir)
    staging_seconds = time.time() - t0
    print(f"staging completed in {staging_seconds:.1f}s")

Mounted at /content/drive
copying /content/drive/MyDrive/DATASET_SHARP/doppler_traces.zip -> /content/doppler_traces.zip
copying /content/drive/MyDrive/DATASET_SHARP/doppler_traces_S4_S5.zip -> /content/doppler_traces_S4_S5.zip
staging completed in 51.7s


## 3. Smoke run: C1 config, reduced horizon

A **copy** of `configs/c1_ce.yaml` — the file is never edited. Only
the horizon shrinks (2 epochs); batch size and steps/epoch stay real
so the measured s/step is the real one. Epoch 1 includes disk-cache
warming; **epoch 2 is the warm number the gate uses**.

Run in two calls on purpose: the second call must log
`resumed C1_smoke ... at epoch 2` — that is the Colab-disconnect
recovery path (§8.2) exercised for real, on a Drive checkpoint.
(Note: bit-exact resume equivalence was verified in the local test
suite; extending max_epochs changes the cosine horizon, §8.1, which
is irrelevant here.)

In [3]:
from sharp_har.train import train_run

cfg = read_yaml(REPO_DIR / "configs" / "c1_ce.yaml")

smoke_cfg = copy.deepcopy(cfg)
smoke_cfg["name"] = "C1_smoke"
smoke_cfg["train"]["max_epochs"] = 1

CKPT_ROOT = Path(paths_cfg["ckpt_root"])

out1 = train_run(smoke_cfg, stage_dir=stage_dir, ckpt_dir=CKPT_ROOT, repo_dir=REPO_DIR)
print("epoch 1 (cold cache):", out1["history"][-1])

2026-07-16 09:17:56,362 [INFO] sharp_har.data: train: 81 traces, 53400 (window, antenna) samples (win=340, stride=100)
2026-07-16 09:17:56,376 [INFO] sharp_har.data: val: 9 traces, 1396 (window, antenna) samples (win=340, stride=340)
2026-07-16 09:21:54,249 [INFO] sharp_har.train: C1_smoke epoch 1/1: loss 1.5358, val macro-F1 0.1092, 0.571 s/step


epoch 1 (cold cache): {'epoch': 1, 'train_loss': 1.5357510292530059, 'val_macro_f1': 0.10918016691212566, 'lr': 0.00020050000000000002, 's_per_step': 0.5712154525518417, 'epoch_seconds': 228.4861810207367}


In [4]:
# Second call, horizon 2: MUST log "resumed C1_smoke ... at epoch 2".
smoke_cfg2 = copy.deepcopy(smoke_cfg)
smoke_cfg2["train"]["max_epochs"] = 2

out2 = train_run(smoke_cfg2, stage_dir=stage_dir, ckpt_dir=CKPT_ROOT, repo_dir=REPO_DIR)
assert len(out2["history"]) == 2, "resume did not continue from the Drive checkpoint"
print("epoch 2 (warm cache):", out2["history"][-1])

run_dir = out2["run_dir"]
for f in ("last.ckpt", "best.ckpt", "run_meta.json", "history.csv"):
    assert (run_dir / f).exists(), f
print("checkpoint artifacts on Drive OK:", run_dir)
print((run_dir / "history.csv").read_text())

2026-07-16 09:22:19,749 [INFO] sharp_har.data: train: 81 traces, 53400 (window, antenna) samples (win=340, stride=100)
2026-07-16 09:22:19,760 [INFO] sharp_har.data: val: 9 traces, 1396 (window, antenna) samples (win=340, stride=340)
2026-07-16 09:22:19,925 [INFO] sharp_har.train: resumed C1_smoke from /content/drive/MyDrive/sharp_har_runs/C1_smoke/last.ckpt at epoch 2
2026-07-16 09:25:50,527 [INFO] sharp_har.train: C1_smoke epoch 2/2: loss 1.2893, val macro-F1 0.2964, 0.521 s/step


epoch 2 (warm cache): {'epoch': 2, 'train_loss': 1.2892506778240205, 'val_macro_f1': 0.2963738122621101, 'lr': 0.00040050000000000003, 's_per_step': 0.5209281605482101, 'epoch_seconds': 208.37126421928406}
checkpoint artifacts on Drive OK: /content/drive/MyDrive/sharp_har_runs/C1_smoke
epoch,train_loss,val_macro_f1,lr,s_per_step,epoch_seconds
1,1.5357510292530059,0.10918016691212566,0.00020050000000000002,0.5712154525518417,228.4861810207367
2,1.2892506778240205,0.2963738122621101,0.00040050000000000003,0.5209281605482101,208.37126421928406



## 4. Throughput gate — measured numbers vs pre-committed rules

Phase-A per-step cost is approximated as **2× the CE step** (512
SupCon views vs 256 windows, §4.2) — declared approximation, to be
replaced by the measured full-batch number on day 3.

In [5]:
s_per_step_cold = out2["history"][0]["s_per_step"]
s_per_step_warm = out2["history"][1]["s_per_step"]

c1_hours = 40 * 400 * s_per_step_warm / 3600
phase_a_hours = 60 * 400 * (2 * s_per_step_warm) / 3600

gate = {
    "git_hash": get_git_hash(REPO_DIR),
    "gpu": torch.cuda.get_device_name(0),
    "staging_seconds": staging_seconds,
    "s_per_step_cold": s_per_step_cold,
    "s_per_step_warm": s_per_step_warm,
    "projected_c1_hours": round(c1_hours, 2),
    "projected_phase_a_hours": round(phase_a_hours, 2),
    "rule_c1_max_hours": 4,
    "rule_phase_a_max_hours": 8,
    "rule_staging_max_seconds": 900,
}
gate["c1_pass"] = c1_hours <= 4
gate["phase_a_pass"] = phase_a_hours <= 8
gate["staging_pass"] = None if staging_seconds is None else staging_seconds <= 900
gate["go"] = bool(gate["c1_pass"] and gate["phase_a_pass"] and gate["staging_pass"] is not False)

print(f"s/step: cold {s_per_step_cold:.3f}, warm {s_per_step_warm:.3f}")
print(f"projected C1: {c1_hours:.2f} h (rule: <= 4 h) -> {'PASS' if gate['c1_pass'] else 'FAIL'}")
print(f"projected phase A (~2x/step): {phase_a_hours:.2f} h (rule: <= 8 h) -> {'PASS' if gate['phase_a_pass'] else 'FAIL'}")
if staging_seconds is not None:
    print(f"staging: {staging_seconds:.0f}s (rule: <= 900s) -> {'PASS' if gate['staging_pass'] else 'FAIL: repack fp16 (§8.5)'}")
else:
    print("staging: not measured this session — measure on a fresh runtime before signing the gate off")
print()
print("GATE:", "GO" if gate["go"] else "NO-GO -> escalation §5.2: (a) 400->300 steps/epoch, (b) escalation_b, (c) shorter horizon")

write_json(REPO_DIR / "reports" / "gate_day2.json", gate)
print("written:", REPO_DIR / "reports" / "gate_day2.json")

s/step: cold 0.571, warm 0.521
projected C1: 2.32 h (rule: <= 4 h) -> PASS
projected phase A (~2x/step): 6.95 h (rule: <= 8 h) -> PASS
staging: 52s (rule: <= 900s) -> PASS

GATE: GO
written: /content/sharp-har/reports/gate_day2.json


## 5. Closing the gate

1. Commit `reports/gate_day2.json` — the written go/no-go (§10.1) —
   and update `STATUS.md` in the same commit.
2. The `C1_smoke` folder on Drive is scratch: delete it. The real C1
   run starts fresh, from the unmodified `c1_ce.yaml`, only after GO.
3. If NO-GO: apply the escalation, re-run THIS notebook, and record
   the escalation in the split file changelog (§5.2).
4. Recalibrate the §8.4 budget table with the measured numbers.